# Chapter 2: Lie群上のPose Graph最適化

**SLAM Handbook Section 2.1.3–2.1.5** — SE(2)上のGauss-Newton法を実装し、Lie群上の不確かさも可視化します。

## このNotebookで学ぶこと

1. **SE(2)上のPose Graph SLAM** — Lie群の⊕/⊖を使ったGauss-Newton法
2. **Lie代数でのヤコビアン** — 接空間$\boldsymbol{\xi}$に対する微分
3. **不確かさの表現** — Lie群上のガウス分布 $\mathbf{T} = \bar{\mathbf{T}} \, \text{Exp}(\boldsymbol{\delta})$, $\boldsymbol{\delta} \sim \mathcal{N}(\mathbf{0}, \boldsymbol{\Sigma})$
4. **Adjoint表現** — 摂動の左右変換

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.linalg import expm, logm

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

def normalize_angle(a):
    return (a + np.pi) % (2 * np.pi) - np.pi

## 2.1 SE(2)の実装

2Dポーズ $(x, y, \theta)$ を3×3同次変換行列で表現します。Ch01では $(x,y,\theta)$ をベクトルとして加算していましたが、ここでは正しくLie群として扱います。

$$\mathbf{T} = \begin{bmatrix} \cos\theta & -\sin\theta & x \\ \sin\theta & \cos\theta & y \\ 0 & 0 & 1 \end{bmatrix} \in \text{SE}(2)$$

Lie代数: $\boldsymbol{\xi} = [\rho_1, \rho_2, \theta]^\top \in \mathbb{R}^3$

In [ ]:
# =============================================================
# SE(2) の完全な実装
# =============================================================

class SE2:
    """SE(2) — 2Dポーズ群 (3×3同次変換行列)"""
    
    @staticmethod
    def from_xytheta(x, y, theta):
        c, s = np.cos(theta), np.sin(theta)
        return np.array([[c, -s, x], [s, c, y], [0, 0, 1]])
    
    @staticmethod
    def x(T): return T[0, 2]
    @staticmethod
    def y(T): return T[1, 2]
    @staticmethod
    def theta(T): return np.arctan2(T[1, 0], T[0, 0])
    @staticmethod
    def t(T): return T[:2, 2]
    @staticmethod
    def R(T): return T[:2, :2]
    
    @staticmethod
    def inv(T):
        R = T[:2, :2]
        t = T[:2, 2]
        T_inv = np.eye(3)
        T_inv[:2, :2] = R.T
        T_inv[:2, 2] = -R.T @ t
        return T_inv
    
    @staticmethod
    def Exp(xi):
        """R^3 → SE(2) 指数写像 (closed-form)"""
        rho, theta = xi[:2], xi[2]
        if abs(theta) < 1e-10:
            return np.array([[1, 0, rho[0]], [0, 1, rho[1]], [0, 0, 1]])
        
        c, s = np.cos(theta), np.sin(theta)
        # V行列 (並進部分の変換)
        V = np.array([[s/theta, -(1-c)/theta],
                       [(1-c)/theta, s/theta]])
        t = V @ rho
        T = np.eye(3)
        T[:2, :2] = np.array([[c, -s], [s, c]])
        T[:2, 2] = t
        return T
    
    @staticmethod
    def Log(T):
        """SE(2) → R^3 対数写像"""
        theta = SE2.theta(T)
        t = SE2.t(T)
        if abs(theta) < 1e-10:
            return np.array([t[0], t[1], theta])
        
        c, s = np.cos(theta), np.sin(theta)
        V = np.array([[s/theta, -(1-c)/theta],
                       [(1-c)/theta, s/theta]])
        rho = np.linalg.solve(V, t)
        return np.array([rho[0], rho[1], theta])
    
    @staticmethod
    def oplus(T, xi):
        """T ⊕ ξ = T · Exp(ξ)"""
        return T @ SE2.Exp(xi)
    
    @staticmethod
    def ominus(T1, T2):
        """T1 ⊖ T2 = Log(T2^{-1} · T1)"""
        return SE2.Log(SE2.inv(T2) @ T1)
    
    @staticmethod
    def Adjoint(T):
        """Adjoint表現 Ad(T): 3×3行列"""
        R = T[:2, :2]
        t = T[:2, 2]
        Ad = np.eye(3)
        Ad[:2, :2] = R
        Ad[0, 2] = t[1]
        Ad[1, 2] = -t[0]
        return Ad

# テスト
T = SE2.from_xytheta(1.0, 2.0, np.pi/4)
xi = SE2.Log(T)
T_recovered = SE2.Exp(xi)

print(f"T =\n{T}")
print(f"Log(T) = {xi}")
print(f"Exp(Log(T)) ≈ T: {np.allclose(T, T_recovered)} ✓")

# ⊕/⊖テスト
xi_small = np.array([0.1, 0.05, 0.03])
T2 = SE2.oplus(T, xi_small)
xi_recovered = SE2.ominus(T2, T)
print(f"\nξ: {xi_small}")
print(f"T2⊖T: {xi_recovered}")
print(f"一致: {np.allclose(xi_small, xi_recovered)} ✓")

## 2.2 SE(2)上のPose Graph SLAM

Ch01ではポーズを $(x,y,\theta)$ ベクトルとして $\mathbf{x} \leftarrow \mathbf{x} + \boldsymbol{\delta}$ で更新しましたが、ここでは正しく:

$$\mathbf{T}_i \leftarrow \mathbf{T}_i \oplus \boldsymbol{\xi}_i^* = \mathbf{T}_i \, \text{Exp}(\boldsymbol{\xi}_i^*)$$

で更新します。

### Pose Graphの残差

相対ポーズ計測 $\mathbf{Z}_{ij}$ に対する残差を **Lie代数上** で定義（SLAM Handbook Eq. 2.20）：

$$\mathbf{e}_{ij} = \text{Log}(\mathbf{Z}_{ij}^{-1} \, \mathbf{T}_i^{-1} \, \mathbf{T}_j)$$

$\mathbf{e}_{ij} = \mathbf{0}$ なら $\mathbf{T}_i^{-1}\mathbf{T}_j = \mathbf{Z}_{ij}$（完全一致）。

In [ ]:
# =============================================================
# Pose Graph SLAM on SE(2) — 問題設定
# =============================================================
np.random.seed(42)

# 真のポーズ: 正方形の軌跡 (8ポーズ)
poses_true = [
    SE2.from_xytheta(0, 0, 0),
    SE2.from_xytheta(2, 0, 0),
    SE2.from_xytheta(4, 0, np.pi/2),
    SE2.from_xytheta(4, 2, np.pi/2),
    SE2.from_xytheta(4, 4, np.pi),
    SE2.from_xytheta(2, 4, np.pi),
    SE2.from_xytheta(0, 4, -np.pi/2),
    SE2.from_xytheta(0, 2, -np.pi/2),
]
n_poses = len(poses_true)

# ノイズパラメータ
sigma_odom = np.array([0.1, 0.1, 0.02])  # [m, m, rad]
Omega_odom = np.diag(1.0 / sigma_odom**2)  # 情報行列
sigma_loop = np.array([0.15, 0.15, 0.03])
Omega_loop = np.diag(1.0 / sigma_loop**2)

# オドメトリ計測の生成 (連続ポーズ間)
edges = []  # (i, j, Z_ij, Omega_ij)
for i in range(n_poses - 1):
    Z_true = SE2.inv(poses_true[i]) @ poses_true[i+1]  # 真の相対ポーズ
    noise = np.random.normal(0, sigma_odom)
    Z_noisy = SE2.oplus(Z_true, noise)
    edges.append((i, i+1, Z_noisy, Omega_odom))

# ループクロージャ: pose 7 → pose 0 (ループを閉じる)
Z_loop_true = SE2.inv(poses_true[7]) @ poses_true[0]
noise_loop = np.random.normal(0, sigma_loop)
Z_loop_noisy = SE2.oplus(Z_loop_true, noise_loop)
edges.append((7, 0, Z_loop_noisy, Omega_loop))

print(f"ポーズ数: {n_poses}")
print(f"エッジ数: {len(edges)} ({n_poses-1} odom + 1 loop closure)")

# 可視化
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
for i, T in enumerate(poses_true):
    x, y, th = SE2.x(T), SE2.y(T), SE2.theta(T)
    ax.plot(x, y, 'ko', markersize=8)
    ax.arrow(x, y, 0.3*np.cos(th), 0.3*np.sin(th),
             head_width=0.1, fc='black', ec='black')
    ax.text(x+0.15, y+0.15, f'T{i}', fontsize=10)

# ループクロージャを赤で
ax.plot([SE2.x(poses_true[7]), SE2.x(poses_true[0])],
        [SE2.y(poses_true[7]), SE2.y(poses_true[0])],
        'r--', linewidth=2, label='Loop closure')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.legend(); ax.set_title('Ground Truth + Loop Closure', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# =============================================================
# SE(2) Pose Graph SLAM — Gauss-Newton on Lie group
# =============================================================

def numerical_jacobian_se2(func, T, eps=1e-7):
    """SE(2)上の数値ヤコビアン (摂動: T ⊕ εe_k)"""
    e0 = func(T)
    n = len(e0)
    J = np.zeros((n, 3))
    for k in range(3):
        delta = np.zeros(3)
        delta[k] = eps
        T_perturbed = SE2.oplus(T, delta)
        e_perturbed = func(T_perturbed)
        J[:, k] = (e_perturbed - e0) / eps
    return J

def pose_graph_gauss_newton(poses_init, edges, n_iter=20, fix_first=True):
    """SE(2)上のPose Graph最適化 (Gauss-Newton法)
    
    残差: e_ij = Log(Z_ij^{-1} · T_i^{-1} · T_j)
    更新: T_i ← T_i · Exp(ξ_i)
    """
    poses = [T.copy() for T in poses_init]
    n = len(poses)
    history = []
    
    for iteration in range(n_iter):
        dim = 3 * n  # 各ポーズ3自由度
        H = np.zeros((dim, dim))
        b = np.zeros(dim)
        total_cost = 0
        
        for (i, j, Z_ij, Omega) in edges:
            # 残差: e = Log(Z^{-1} · T_i^{-1} · T_j)
            T_rel = SE2.inv(poses[i]) @ poses[j]
            e = SE2.Log(SE2.inv(Z_ij) @ T_rel)
            
            # 数値ヤコビアン
            def residual_wrt_Ti(Ti):
                T_rel_ = SE2.inv(Ti) @ poses[j]
                return SE2.Log(SE2.inv(Z_ij) @ T_rel_)
            
            def residual_wrt_Tj(Tj):
                T_rel_ = SE2.inv(poses[i]) @ Tj
                return SE2.Log(SE2.inv(Z_ij) @ T_rel_)
            
            Ji = numerical_jacobian_se2(residual_wrt_Ti, poses[i])
            Jj = numerical_jacobian_se2(residual_wrt_Tj, poses[j])
            
            # 情報行列付き
            si, ei = 3*i, 3*i+3
            sj, ej = 3*j, 3*j+3
            
            H[si:ei, si:ei] += Ji.T @ Omega @ Ji
            H[si:ei, sj:ej] += Ji.T @ Omega @ Jj
            H[sj:ej, si:ei] += Jj.T @ Omega @ Ji
            H[sj:ej, sj:ej] += Jj.T @ Omega @ Jj
            
            b[si:ei] += Ji.T @ Omega @ e
            b[sj:ej] += Jj.T @ Omega @ e
            
            total_cost += e.T @ Omega @ e
        
        # 最初のポーズを固定（ゲージ自由度の除去）
        if fix_first:
            H[:3, :] = 0; H[:, :3] = 0
            H[:3, :3] = np.eye(3) * 1e6
            b[:3] = 0
        
        # 解く
        xi = np.linalg.solve(H, -b)
        xi_norm = np.linalg.norm(xi)
        history.append((total_cost, xi_norm))
        
        # 更新: T_i ← T_i · Exp(ξ_i)
        for i in range(n):
            poses[i] = SE2.oplus(poses[i], xi[3*i:3*i+3])
        
        if xi_norm < 1e-6:
            print(f"収束! (iteration {iteration+1})")
            break
    
    return poses, history, H

print("Pose Graph solver on SE(2) 定義完了")

In [ ]:
# =============================================================
# 実行: オドメトリ累積 → Gauss-Newton最適化
# =============================================================

# 初期推定: オドメトリを累積
poses_init = [poses_true[0].copy()]  # 最初のポーズは固定
for i, j, Z_ij, _ in edges[:-1]:  # ループクロージャ以外
    poses_init.append(poses_init[-1] @ Z_ij)

# 最適化
poses_opt, history, H_final = pose_graph_gauss_newton(poses_init, edges)

# 結果表示
print(f"\n=== 結果 ===")
for i in range(n_poses):
    x_opt = SE2.x(poses_opt[i])
    y_opt = SE2.y(poses_opt[i])
    th_opt = np.degrees(SE2.theta(poses_opt[i]))
    x_true = SE2.x(poses_true[i])
    y_true = SE2.y(poses_true[i])
    err = np.sqrt((x_opt-x_true)**2 + (y_opt-y_true)**2)
    print(f"  T{i}: ({x_opt:6.3f}, {y_opt:6.3f}, {th_opt:6.1f}°)  "
          f"真値: ({x_true:6.3f}, {y_true:6.3f})  誤差: {err:.4f}m")

In [ ]:
# =============================================================
# 可視化: Before/After + 不確かさ楕円
# =============================================================

def plot_poses(ax, poses, color='blue', label=None, alpha=1.0):
    xs = [SE2.x(T) for T in poses]
    ys = [SE2.y(T) for T in poses]
    ax.plot(xs, ys, 'o-', color=color, markersize=6, label=label, alpha=alpha)
    for i, T in enumerate(poses):
        th = SE2.theta(T)
        ax.arrow(SE2.x(T), SE2.y(T), 0.25*np.cos(th), 0.25*np.sin(th),
                 head_width=0.08, fc=color, ec=color, alpha=alpha)

def plot_covariance_2d(ax, x, y, cov_2x2, n_std=2, color='blue', alpha=0.2):
    eigenvalues, eigenvectors = np.linalg.eigh(cov_2x2)
    angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
    width = 2 * n_std * np.sqrt(max(eigenvalues[0], 0))
    height = 2 * n_std * np.sqrt(max(eigenvalues[1], 0))
    ellipse = Ellipse((x, y), width, height, angle=angle,
                       fill=True, facecolor=color, alpha=alpha, edgecolor=color)
    ax.add_patch(ellipse)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Before
plot_poses(axes[0], poses_true, 'black', 'Ground Truth', alpha=0.3)
plot_poses(axes[0], poses_init, 'red', 'Odometry (初期推定)')
axes[0].set_title('Before Optimization', fontweight='bold')
axes[0].set_aspect('equal'); axes[0].grid(True, alpha=0.3); axes[0].legend()

# After + 不確かさ楕円
plot_poses(axes[1], poses_true, 'black', 'Ground Truth', alpha=0.3)
plot_poses(axes[1], poses_opt, 'blue', 'Optimized')

# 共分散行列の計算 (H^{-1} の xy部分)
try:
    cov_full = np.linalg.inv(H_final)
    for i in range(n_poses):
        cov_i = cov_full[3*i:3*i+2, 3*i:3*i+2]  # xy部分のみ
        plot_covariance_2d(axes[1], SE2.x(poses_opt[i]), SE2.y(poses_opt[i]),
                          cov_i, n_std=3, color='blue', alpha=0.15)
except np.linalg.LinAlgError:
    print("共分散計算をスキップ")

axes[1].set_title('After Optimization + 3σ Uncertainty', fontweight='bold')
axes[1].set_aspect('equal'); axes[1].grid(True, alpha=0.3); axes[1].legend()

plt.tight_layout()
plt.show()

print("→ ループクロージャにより、全ポーズの不確かさが減少")
print("→ ⊕/⊖演算子でSE(2)の制約を常に満たしたまま最適化")

## 2.3 Lie群上の不確かさの可視化

Lie群上の「ガウス分布」は接空間で定義します（SLAM Handbook Eq. 2.22-2.23）：

$$\mathbf{T} = \bar{\mathbf{T}} \, \text{Exp}(\boldsymbol{\delta}), \quad \boldsymbol{\delta} \sim \mathcal{N}(\mathbf{0}, \boldsymbol{\Sigma})$$

接空間でサンプルした $\boldsymbol{\delta}$ を $\text{Exp}$ でマニフォールドに写像することで、ポーズのばらつきを生成できます。

In [ ]:
# =============================================================
# Lie群上のガウス分布: サンプリングで可視化
# =============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 3つの異なる不確かさパターン
T_mean = SE2.from_xytheta(2, 1, np.pi/4)
covariances = [
    ('等方的', np.diag([0.1, 0.1, 0.01])**2),
    ('位置のみ不確か', np.diag([0.3, 0.05, 0.01])**2),
    ('角度が不確か', np.diag([0.05, 0.05, 0.2])**2),
]

n_samples = 200

for ax, (title, Sigma) in zip(axes, covariances):
    # サンプリング
    samples_x, samples_y = [], []
    for _ in range(n_samples):
        delta = np.random.multivariate_normal([0, 0, 0], Sigma)
        T_sample = SE2.oplus(T_mean, delta)
        samples_x.append(SE2.x(T_sample))
        samples_y.append(SE2.y(T_sample))
        # 方向も描画
        th = SE2.theta(T_sample)
        ax.arrow(SE2.x(T_sample), SE2.y(T_sample),
                 0.1*np.cos(th), 0.1*np.sin(th),
                 head_width=0.02, fc='blue', ec='blue', alpha=0.1)
    
    ax.scatter(samples_x, samples_y, c='blue', s=5, alpha=0.3)
    
    # 平均ポーズ
    xm, ym, thm = SE2.x(T_mean), SE2.y(T_mean), SE2.theta(T_mean)
    ax.plot(xm, ym, 'r*', markersize=15, zorder=5)
    ax.arrow(xm, ym, 0.3*np.cos(thm), 0.3*np.sin(thm),
             head_width=0.06, fc='red', ec='red', zorder=5)
    
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_title(f'{title}\nΣ = diag{np.sqrt(np.diag(Sigma)).round(2).tolist()}',
                 fontsize=11, fontweight='bold')

plt.suptitle('SE(2)上のガウス分布: T = T̄ · Exp(δ), δ ~ N(0, Σ)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2.4 演習問題

### 演習1: Ch01との比較
Ch01の `02_nonlinear_slam.ipynb` のGauss-Newton法（ベクトル加算更新）と、このNotebookのLie群更新を同じ問題で比較して、結果の違いを観察してください。大きな回転がある場合に差が出ますか？

### 演習2: ループクロージャの追加
`edges` にさらにループクロージャ（例: T3→T7）を追加して、不確かさ楕円がどう縮小するか確認してください。

### 演習3: SE(3)への拡張
このNotebookのSE(2) Pose Graph SolverをSE(3)に拡張してみましょう。状態は6自由度、ヤコビアンは6×6になります。

## まとめ

- **Lie群上の最適化**: 更新を $\mathbf{T} \leftarrow \mathbf{T} \cdot \text{Exp}(\boldsymbol{\xi})$ で行うことで制約を常に満たす
- **残差はLie代数上で定義**: $\mathbf{e}_{ij} = \text{Log}(\mathbf{Z}_{ij}^{-1} \mathbf{T}_i^{-1} \mathbf{T}_j)$
- **不確かさ**: 接空間でのガウス分布 $\boldsymbol{\delta} \sim \mathcal{N}(\mathbf{0}, \boldsymbol{\Sigma})$ をExp写像でマニフォールドへ
- **数値ヤコビアン**: SE(2)上の摂動 $\mathbf{T} \oplus \epsilon \mathbf{e}_k$ で計算（解析ヤコビアンも可能）

### 次のNotebook
→ Ch03: 外れ値対策（RANSAC, Robust Cost Functions）